# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

# Teoria:


Proces - instancja wykonywanego programu. Każdy ma jakieś zasoby i wykonuje instrukcje programu. W Pythonie każdy proces jest w rzeczywistości osobną instancją interpretera.

Wątek to osobna jednostka, która działa w obrębie tego samego procesu. Oznacza to, że najprostsze programy maja proces i co najmniej jeden wątek. Watki w Pythonie są rzeczywistymi wątkami, nie ma wirtualizacji.

Wątek ma stany: utworzenia, wykonywania, zablokowania (opcjonalnie), zakończenia

Równoległość - wiele zadań w tym samym czasie za pomocą wielu procesorów lub przedni procesora 
Współbieżność - techniki organizacji programu, które dopuszcza istnienie wielu wątków, przeplecione ze sobą
W Pythonie podstawowym modułem do programowania współbieżnego jest threading, natomiast do programowania równoległego multiprocessing.

Asynchronicznosc - paradygmat, który umożliwia wspobiezne i nieblokujące wykonywanie zadań w aplikacji. W Pythonie implementacja przez moduł syncio.

Różnica jest taka, że w ramach threading wielozadaniowość jest z wypłaszczeniem atencji procesora, w asyncio mamy wielozadaniowość pooperacyjną, poszczególne zadania musza ogłosić że są gotowe do wyłączenia

W I/O bond głównym ograniczeniem wydajniościowym jest czas oczekiwania na operacje wejścia wyjścia.

Współbieżność i asynchronicznosc: serwery sieciowe, interaktywne aplikacje, integracja z bazami danych; tam gdzie trzeba poczekać - zewnętrzne programy muszą się połączyć
Równoległość: operacje na dużych zbiorach danych, obliczenia numeryczne; mamy juz dane ale ciężkie i dużo danych, ograniczeniem jest CPU bound i robimy multiprocessing

Multiprocessing odnosi się do zdolności systemu do obsługi więcej niż jednego procesora
jednocześnie. Aplikacje w systemie wieloprocesorowym są "podzielone" na mniejsze, które
działają niezależnie. System operacyjny przydziela te wątki do procesorów, poprawiając
wydajność systemu.

GIL - global interpreter lock jest blokadą (lock), która pozwala tylko jednemu wątkowi kontrolować interpreter Pythona co eliminuje potrzebę synchronizacji dostępu do wrażliwych obszarów pamięci. GIL to pojedyncza blokada na samym interpreterze, która wprowadza zasadę, ze wykonanie dowolnego kodu pythona wymaga uzyskanie blokady interpretera, zapobiega to zakleszczeniom. Można kompensować utratę korzyści z wydajności pojedynczego wątku wynikająca z GIL, dodając inne funkcje zwiększające wydajność, takie jak kompliatoru JIT.

Modul threading w python jest wbudowanym modułem, który umożliwia programowanie wielowątkowe. Implementuje funkcjonalności: tworzenie wątków, zarządzanie watkami, synchronizacji, obsługi wyjątków.
Tworzenie nowych wątków za pomocą klasy thread. Należy określić docelową funkcję wykonywaną w wątku (target). 
Wątki Daemon w Pythonie są zamykane automatycznie, gdy główny watek programu zakończy swoje działanie. Jeśli program uruchamia wątki, które nie są watkami daemon to program będzie oczekiwać aż te watki zakończą swoje działanie przed zakończeniem pracy programu. 

ThreadPoolExecutor to część modułu concurrent.futures w Pythonie, która zapewnia
interfejs do tworzenia i zarządzania pulą wątków dla równoległego wykonywania zadań.
Określamy rozmiar puli wątków podczas tworzenia instancji ThreadPoolExecutor.
Możemy przekazać funkcje lub wyrażenia do wykonania do puli wątków za pomocą metody
submit() lub map().
Automatycznie zarządza cyklem życia wątków w puli, takimi jak startowanie, zatrzymywanie i ponowne uruchamianie.
Zaleca się korzystanie z ThreadPoolExecutor z użyciem menedżera kontekstu (with),
aby nie wywoływać .join() dla wątków lub .shutdown() w samej puli.

Race condition to sytuacja, w której wynik działania programu zależy od kolejności lub
czasu wykonania różnych wątków lub procesów. W Pythonie, race conditions mogą wystąpić,
gdy kilka wątków próbuje równocześnie modyfikować współdzielone zasoby bez
odpowiedniej synchronizacji.

Lock jest mechanizmem synchronizacji, który pozwala kontrolować dostęp wielu wątków do
współdzielonych zasobów. Jest obiektem, który działa jak przepustka do zasobów.
Każdy wątek, który chce przepustkę, musi poczekać, aż zostanie zwolniona.
Metoda.acquire() na Lock w wątku tworzy "blokadę", a.release() ją zwalnia.
Lock może działać jako menadżer konteksu, więc można go używać z with.

Deadlock (zakleszczenie) to sytuacja, gdy dwa lub więcej wątków są zablokowane
wzajemnie i oczekują na siebie nawzajem, aby zwolnić zasoby, które trzymają, przed
kontynuacją swojego działania.
W rezultacie żaden z wątków lub procesów nie może kontynuować pracy, co prowadzi do
zatrzymania się całego systemu.

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [1]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Pytia ‘44. Prezentacja książki Jarosława Księżyka
2. Pokazy filmowe w Centrum Kultury Żydowskiej
3. Poranki organowe w Filharmonii Krakowskiej
4. Muzyczny koktajl
5. Jan Józef Szczepański na czas nie-pokoju
6. Spotkania autorskie Wydawnictwa Austeria
7. Burlesque Show
8. Zdzisław Beksiński. Rysunki
9. 24. Tydzień Kina Hiszpańskiego
10. Szapocznikow. Osobista

Czas wykonania: 3.34s


In [2]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Pytia ‘44. Prezentacja książki Jarosława Księżyka
2. Pokazy filmowe w Centrum Kultury Żydowskiej
3. Poranki organowe w Filharmonii Krakowskiej
4. Muzyczny koktajl
5. Jan Józef Szczepański na czas nie-pokoju
6. Spotkania autorskie Wydawnictwa Austeria
7. Burlesque Show
8. Zdzisław Beksiński. Rysunki
9. 24. Tydzień Kina Hiszpańskiego
10. Szapocznikow. Osobista

Czas wykonania (wątki): 0.82s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [ ]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock: # watek probuje zlapac locka, tylko jeden na raz, po wyjsciu zwalniany dla kolejnych
                        # gdy zakomentuje locka, to kwota koncowa przestaje byc =100
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [ ]:
# funkcje importowane:

def is_prime(n):
    if n <= 1: return False
    if n == 2: return True
    if n % 2 == 0: return False
    i = 3
    while i * i <= n:
        if n % i == 0: return False
        i += 2
    return True

def find_primes(start, end):
    return [num for num in range(start, end) if is_prime(num)]

def calculate_power_sum(n):
    return sum(n**i for i in range(1, 101))

In [28]:
import time

start = time.time()
find_primes(0, 1000000)
print(f'Czas wykonania = {time.time() - start:.4f} sek')

Czas wykonania = 1.7059 sek


In [21]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 8 procesach (rdzeniach)...
Zakończono w czasie 0.59s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [ ]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"
fakty = []

start = time.time()

def get_fact():
    return requests.get(CAT_API_URL).json().get('fact')

for i in range(20):
    fakty.append(get_fact())

print('\n'.join(fakty))
print(f'\n\nCzas wykonania wyniósł: {time.time() - start:.2f}sek')

Normal body temperature for a cat is 102 degrees F.
According to a Gallup poll, most American pet owners obtain their cats by adopting strays.
Milk can give some cats diarrhea.
Blue-eyed, white cats are often prone to deafness.
The cat has 500 skeletal muscles (humans have 650).
Neutering a male cat will, in almost all cases, stop him from spraying (territorial marking), fighting with other males (at least over females), as well as lengthen his life and improve its quality.
Edward Lear, author of \The Owl and the Pussycat\"", is said to have had his new house in San Remo built to exactly the same specification as his previous residence, so that his much-loved tabby, Foss, would immediately feel at home."""
Cats spend nearly 1/3 of their waking hours cleaning themselves.
The heaviest cat on record is Himmy, a Tabby from Queensland, Australia. He weighed nearly 47 pounds (21 kg). He died at the age of 10.
Cats have 30 teeth (12 incisors, 10 premolars, 4 canines, and 4 molars), while dogs

In [ ]:
start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    futures = [executor.submit(get_fact) for _ in range(20)]
    # zeby uzyc map() musialabym zrobic funkcje z argumentem (_) udawanym
    results = [f.result() for f in futures]

print('\n'.join(results))
print(f'\n\nCzas wykonania wyniósł: {time.time() - start:.2f}sek')

An adult lion's roar can be heard up to five miles (eight kilometers) away.
The average cat can jump 8 feet in a single bound, nearly six times its body length!
The Amur leopard is one of the most endangered animals in the world.
A cat's appetite is the barometer of its health. Any cat that does not eat or drink for more than two days should be taken to a vet.
Retractable claws are a physical phenomenon that sets cats apart from the rest of the animal kingdom. I n the cat family, only cheetahs cannot retract their claws.
The cheetah is the world's fastest land mammal. It can run at speeds of up to 70 miles an hour (113 kilometers an hour).
A group of cats is called a “clowder.”
Milk can give some cats diarrhea.
A cat lover is called an Ailurophilia (Greek: cat+lover).
A cat can travel at a top speed of approximately 31 mph (49 km) over a short distance.
The group of words associated with cat (catt, cath, chat, katze) stem from the Latin catus, meaning domestic cat, as opposed to feles,

### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [ ]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time

# Twój kod tutaj...
limit_numbers = 20

queue_p = queue.Queue()
queue_np = queue.Queue()

def producent():
    for i in range(1, limit_numbers + 1):
        if i % 2 == 0:
            print(f"[Producent] -> parzyste: {i}")
            queue_p.put(i)
        else:
            print(f"[Producent] -> nieparzyste: {i}")
            queue_np.put(i)
        
        time.sleep(0.01)
    
    # koniec dla obu
    queue_p.put(None)
    queue_np.put(None)

def konsument_parzysty():
    while True:
        item = queue_p.get()
        
        if item is None:
            break
        
        print(f"[Konsument parzysty] Wziął: {item}")
        queue_p.task_done()

def konsument_nieparzysty():
    while True:
        item = queue_np.get()
        
        if item is None:
            break
        
        print(f"[Konsument nieparzysty] Wziął: {item}")
        queue_np.task_done()

# wątki
t_producent = threading.Thread(target=producent)
t_konsument_parzysty = threading.Thread(target=konsument_parzysty)
t_konsument_nieparzysty = threading.Thread(target=konsument_nieparzysty)

# start - wszystkie operacje na raz
t_producent.start()
t_konsument_parzysty.start()
t_konsument_nieparzysty.start()

# czekamy
t_producent.join()
t_konsument_parzysty.join()
t_konsument_nieparzysty.join()

print("====== Koniec ======")

[Producent] -> nieparzyste: 1
[Konsument nieparzysty] Wziął: 1
[Producent] -> parzyste: 2
[Konsument parzysty] Wziął: 2
[Producent] -> nieparzyste: 3
[Konsument nieparzysty] Wziął: 3
[Producent] -> parzyste: 4
[Konsument parzysty] Wziął: 4
[Producent] -> nieparzyste: 5
[Konsument nieparzysty] Wziął: 5
[Producent] -> parzyste: 6
[Konsument parzysty] Wziął: 6
[Producent] -> nieparzyste: 7
[Konsument nieparzysty] Wziął: 7
[Producent] -> parzyste: 8
[Konsument parzysty] Wziął: 8
[Producent] -> nieparzyste: 9
[Konsument nieparzysty] Wziął: 9
[Producent] -> parzyste: 10
[Konsument parzysty] Wziął: 10
[Producent] -> nieparzyste: 11
[Konsument nieparzysty] Wziął: 11
[Producent] -> parzyste: 12
[Konsument parzysty] Wziął: 12
[Producent] -> nieparzyste: 13
[Konsument nieparzysty] Wziął: 13
[Producent] -> parzyste: 14
[Konsument parzysty] Wziął: 14
[Producent] -> nieparzyste: 15
[Konsument nieparzysty] Wziął: 15
[Producent] -> parzyste: 16
[Konsument parzysty] Wziął: 16
[Producent] -> nieparzyste

### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [51]:
start = time.time()
[calculate_power_sum(i) for i in range(100000)]
print(f'Zakończono w czasie {time.time() - start:.4f} sek')

Zakończono w czasie 6.1962 sek


In [52]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

def run_power_sum():
    cores = multiprocessing.cpu_count()
    start = time.time()

    numbers = [i for i in range(100000)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.map(calculate_power_sum, numbers)
    
    #print(results)
    print(f'Zakończono obliczenia na {cores} rdzeniach w czasie {time.time() - start:.4f} sek')

if __name__ == "__main__":
    run_power_sum()

Zakończono obliczenia na 8 rdzeniach w czasie 1.2852 sek
